# KBO deep_learning_state — Colab runner

Thin wrapper. No training logic lives in this notebook; every cell shells out to
`src/deep_learning_state/`. Edit the code on the Mac, push, re-run here.

**Runtime > Change runtime type > GPU** before running anything.

## 1. Clone / pull

In [ ]:
REPO = 'https://github.com/Gromiit/kbo-control.git'
import os, pathlib
if pathlib.Path('/content/kbo/.git').exists():
    !cd /content/kbo && git pull --ff-only
else:
    !git clone $REPO /content/kbo
os.chdir('/content/kbo')
!git rev-parse --short HEAD


## 2. Dependencies + CUDA check

In [ ]:
!bash scripts/setup_colab.sh

## 3. Attach the shards

Shards are built on the Mac and are NOT in git. They arrive as one tarball,
`kbo_seq_L32.tgz` (198 MB compressed, 2.2 GB extracted).

**Extract to /content, do not symlink or mmap out of Drive.** The dataset
memory-maps `.npy`; mmap over the Drive FUSE mount is a network round trip per
page and turns a 40-second epoch into tens of minutes. /content has ~70 GB.

Checkpoints and results go the other way -- straight to Drive via
`KBO_CKPT` / `KBO_EXP`, so a disconnect costs nothing.

Override the archive location with `KBO_DATA_ARCHIVE` if it is not at the
default `MyDrive/kbo/kbo_seq_L32.tgz`.


In [ ]:
import os
from google.colab import drive; drive.mount('/content/drive')

os.environ.setdefault('KBO_DATA_ARCHIVE',
                      '/content/drive/MyDrive/kbo/kbo_seq_L32.tgz')
# optional: sums written next to the Mac build, uploaded alongside the tarball
os.environ.setdefault('KBO_DATA_SHA256',
                      '/content/drive/MyDrive/kbo/SHA256SUMS_L32.txt')

# shards: Drive -> local disk. macOS tar ships `._*` sidecars; they are inert
# (the shard glob is anchored, so it never matches them) and cost ~30 KB.
!mkdir -p /content/kbo/data
!tar xzf "$KBO_DATA_ARCHIVE" -C /content/kbo/data
!du -sh /content/kbo/data/sequences && ls /content/kbo/data/sequences

# checkpoints + results stay on Drive
os.environ['KBO_CKPT'] = '/content/drive/MyDrive/kbo/out/checkpoints'
os.environ['KBO_EXP']  = '/content/drive/MyDrive/kbo/out'
!mkdir -p "$KBO_CKPT"


### 3b. Verify what arrived

Inventory, shapes, and the window invariants that are visible in the shards
alone (left-padding, `length`, p_v9 in valid only). If `SHA256SUMS_L32.txt`
was uploaded it also proves these are byte-identical to the tree the leakage
audit passed on the Mac.

`audit.py` itself is a **Mac** step -- it rebuilds every window from
`data/folds/*.parquet` to compare, and those 1.5 GB of folds are not uploaded.
Season isolation, row_id overlap, window-vs-parquet equality, scaler
provenance and p_v9-vs-OOF alignment were established there before upload.


In [ ]:
import os, pathlib
args = ['--seasons', '2023,2024', '--sequence-length', '32']
sums = os.environ.get('KBO_DATA_SHA256', '')
if pathlib.Path(sums).exists():
    args += ['--sha256', sums]
else:
    print(f'no checksum file at {sums!r} -- structural checks only')
ARGS = ' '.join(args)
!python -m src.deep_learning_state.check_shards $ARGS


## 4. Smoke test (always first)

In [ ]:
!SMOKE_ONLY=1 bash scripts/train_colab.sh

## 5. Full training

Only after the smoke test passes **and** you have been told to run it.
Seeds run sequentially — one GPU, one model at a time.

In [ ]:
!CONFIG=configs/gru_full.yaml SEEDS='42' bash scripts/train_colab.sh

## 6. Results

`KBO_EXP` already points at Drive, so `results.csv`, the per-epoch traces and
the checkpoints are written there directly. Nothing to copy.


In [ ]:
!ls -la "$KBO_EXP"
!cat "$KBO_EXP/results.csv"


## 7. Full Training Remaining Sweep

Cell 5 runs fold 2024 / seed 42 only. This picks up the five runs that are
left, without retraining what is already done:

| fold | seeds | runs |
|---|---|---|
| 2023 | 42, 43, 44 | 3 |
| 2024 | 43, 44 | 2 |

Nothing here retrains fold 2024 / seed 42 — no invocation names that seed for
that fold. Checkpoints are tagged `{name}_fold{fold}_s{seed}`, so the five new
runs write five new pairs of files and the existing
`gru_full_fold2024_s42_{best,last}.pt` is never opened for writing. The cell
after the sweep proves that rather than asserting it.

Two things to expect:

* `train_colab.sh` always smokes before it sweeps, once per fold. That is two
  extra ~10 s runs named `colab_smoke`, and they DO overwrite
  `colab_smoke_fold*_s42_*.pt` from the earlier smoke. Those are wiring
  proofs, not results; no `gru_full_*` checkpoint is touched.
* `KBO_EXP` / `KBO_CKPT` are already set by cell 3, so `results.csv` and the
  checkpoints accumulate in Drive. Re-running cell 3 is not needed and not
  wanted here.

Expect roughly an hour: ~24 s/epoch on the 1.2 M-row fold, 20 epochs, five
runs, less if `patience: 6` fires.


In [ ]:
# --- pre-flight: git / GPU / dataset / existing checkpoints -----------------
import os, pathlib, json, subprocess

CKPT = pathlib.Path(os.environ['KBO_CKPT'])
EXP  = pathlib.Path(os.environ['KBO_EXP'])
SEQ  = pathlib.Path('/content/kbo/data/sequences')

print('=== git ===')
!git -C /content/kbo log --oneline -1
!git -C /content/kbo status --porcelain || true

print('\n=== GPU ===')
!nvidia-smi --query-gpu=name,memory.total,memory.used,temperature.gpu --format=csv

print('\n=== dataset ===')
for S in (2023, 2024):
    m = json.loads((SEQ / f'manifest_S{S}_L32.json').read_text())
    for split, info in m['splits'].items():
        d = SEQ / split / f'S{S}_L32'
        n = len(sorted(d.glob('shard_*_y.npy')))
        print(f'  fold {S} {split:5s}  {info["rows"]:>9,} rows  '
              f'{n:2d}/{info["shards"]:2d} shards  L={info["L"]}  '
              f'static={info["n_static"]}  {"OK" if n == info["shards"] else "MISMATCH"}')
!du -sh /content/kbo/data/sequences

print('\n=== output roots (must be Drive) ===')
print('  KBO_EXP ', EXP)
print('  KBO_CKPT', CKPT)

# snapshot every checkpoint so the post-run cell can prove nothing was clobbered
BEFORE = {p.name: (p.stat().st_size, p.stat().st_mtime_ns)
          for p in sorted(CKPT.glob('*.pt'))}
print(f'\n=== existing checkpoints ({len(BEFORE)}) ===')
for k in BEFORE:
    print('  ', k)


In [ ]:
# --- the sweep: fold 2023 x 3 seeds, then fold 2024 x 2 seeds ---------------
# One shell line with && so a failure in the first stops the second instead of
# burning half an hour on a run whose fold is already known to be broken.
# Output streams live; %%bash would buffer it until the end.
!CONFIG=configs/gru_full.yaml FOLDS='2023' SEEDS='42 43 44' bash scripts/train_colab.sh && CONFIG=configs/gru_full.yaml FOLDS='2024' SEEDS='43 44' bash scripts/train_colab.sh


In [ ]:
# --- verify: coverage, and that nothing existing was overwritten ------------
import pandas as pd, pathlib, os

CKPT = pathlib.Path(os.environ['KBO_CKPT'])
EXP  = pathlib.Path(os.environ['KBO_EXP'])

r = pd.read_csv(EXP / 'results.csv')
g = r[r.run.str.startswith('gru_full')]
print(f'=== gru_full runs in results.csv: {len(g)} ===')
cols = [c for c in ('fold', 'seed', 'epoch', 'Resolution', 'dResolution',
                    'BSS', 'dBSS', 'selection', 'device') if c in g.columns]
print(g[cols].sort_values(['fold', 'seed']).to_string(index=False))

WANT = {(2023, 42), (2023, 43), (2023, 44),
        (2024, 42), (2024, 43), (2024, 44)}
got  = {(int(f), int(s)) for f, s in g[['fold', 'seed']].values}
print('\n완료:', sorted(got))
print('누락:', sorted(WANT - got) or '없음')
print('중복 (fold,seed):',
      {k: v for k, v in g.groupby(['fold', 'seed']).size().items() if v > 1} or '없음')

print('\n=== checkpoints ===')
after = {p.name: (p.stat().st_size, p.stat().st_mtime_ns)
         for p in sorted(CKPT.glob('*.pt'))}
full = sorted(k for k in after if k.startswith('gru_full'))
for k in full:
    print('  ', k)
print(f'  gru_full checkpoints: {len(full)}  (기대 12 = 6 run x best/last)')

# requirement 7: pre-existing gru_full checkpoints must be byte-for-byte untouched
touched = [k for k, v in BEFORE.items()
           if k.startswith('gru_full') and after.get(k) != v]
gone    = [k for k in BEFORE if k not in after]
# the smoke reruns by design, so colab_smoke_* IS expected to change -- shown
# rather than quietly filtered, so the two cases are never confused
smoked  = [k for k, v in BEFORE.items()
           if not k.startswith('gru_full') and after.get(k) != v]
print('\n재작성된 colab_smoke checkpoint (설계상 정상):', smoked or '없음')
print('기존 gru_full checkpoint 덮어써짐:', touched or '없음')
print('삭제된 checkpoint            :', gone or '없음')
print('새로 생긴 checkpoint          :', sorted(set(after) - set(BEFORE)))

ok = not touched and not gone and not (WANT - got)
print('\n' + ('SWEEP COMPLETE — 6/6 (fold, seed) 조합, 기존 파일 무손상'
              if ok else 'CHECK ABOVE — 누락 또는 덮어쓰기 발생'))
